# Guided knee R=8: method comparison

Side-by-side reconstructions per network (one column each), then their residuals, on a few
slices. Reads the per-volume dumps `scripts/dump_eval.py` writes
(`<run>/eval_dump/<volume>.h5`, made by `bigpurple/guided_recon_dump.sbatch`) -- no
networks are rebuilt for runs that have a dump.

A run with **no dump yet** (e.g. the I2SB runs) is computed here on just the chosen slices
(`COMPUTE_MISSING`), through the same code the dump uses (`dump_eval.dump_run` ->
`evaluation/tasks.py`), into `notebooks/_compare_cache/`. That needs torch and, for I2SB's
sampler, a GPU (run this notebook on bigpurple). The measurement is noise-free (val sigma 0),
so those slices see exactly what the dumped runs saw; only I2SB's bridge noise differs from
what a full dump of the same run would draw.

Run top to bottom. Change `METHODS`, `N_SAMPLES` or pin `SAMPLES` in the first cell.

In [ ]:
import os, sys, random
import numpy as np
import h5py
import matplotlib.pyplot as plt

REPO = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, REPO)
os.chdir(REPO)       # run configs hold repo-relative paths (split CSVs, smaps, ...)
RUNS_ROOT = os.environ.get("COMPARE_RUNS_ROOT", os.path.join(
    REPO, "trained_nets", "guided_recon", "knee", "s4_noise-vlow"))

# (column label, run directory name, dataset in its dump). "recon_nodc": I2SB's plain sampler
# (no CG data consistency) on the same bridge noise.
METHODS = [
    ("LPDSNet",       "lpdsnet_R8",                    "recon"),
    ("TGVN",          "tgvn_c6_R8_g1",                 "recon"),
    ("LGGS",          "lggs_R8_g1",                    "recon"),
    ("LGGS guide-W",  "lggs_guide_w_R8_g1",            "recon"),
    ("I2SB",          "sbunet_small_R8_g1_tau0.19",    "recon"),
    ("I2SB + guide",  "sbunet_small_x1_R8_g1_tau0.19", "recon"),
    ("I2SB, no CG",         "sbunet_small_R8_g1_tau0.19",    "recon_nodc"),
    ("I2SB + guide, no CG", "sbunet_small_x1_R8_g1_tau0.19", "recon_nodc"),
]
N_SAMPLES = 4
SAMPLES = None            # or pin them: [("file1000071", 15), ...]  (volume, source slice)
SAMPLE_SEED = 0           # which slices, when SAMPLES is None
COMPUTE_MISSING = True    # runs with no dump: compute the chosen slices here (torch + GPU)
CKPT = "net_bestval.ckpt" # the dumps' checkpoint (guided_recon_dump.sbatch default)
SEED = 1234               # the dumps' seed (guided_recon_dump.sbatch default)
FLIP = True               # the stored knee slices are upside down
CROP = True               # crop to the anatomy (drops the readout oversampling)
RESID_GAIN = 4.0          # residuals are shown x this
CACHE = os.environ.get("COMPARE_CACHE", os.path.join(REPO, "notebooks", "_compare_cache"))
print("runs under", RUNS_ROOT)

In [ ]:
def _dirs(run):
    """Where a run's volumes may be: its dump, then this notebook's cache."""
    return [d for d in (os.path.join(RUNS_ROOT, run, "eval_dump"),
                        os.path.join(CACHE, run, "eval_dump")) if os.path.isdir(d)]


def find(run, vol, sl):
    """(file, row) holding source slice `sl` of `vol` for `run`, or None."""
    for d in _dirs(run):
        p = os.path.join(d, vol + ".h5")
        if os.path.exists(p):
            with h5py.File(p, "r") as f:
                idx = [int(s) for s in f["slice_index"][()]]
            if sl in idx:
                return p, idx.index(sl)
    return None


def read(run, vol, sl, ds="recon"):
    """One slice of one run: the image `ds`, the shared images, its stored metrics, attrs."""
    p, i = find(run, vol, sl)
    with h5py.File(p, "r") as f:
        out = {k: f[k][i] for k in ("reference", "zero_filled", "guide", "organ_mask") if k in f}
        out["img"] = f[ds][i]
        suffix = "" if ds == "recon" else "_" + ds[len("recon_"):]
        out["metrics"] = {m: float(f[f"{m}{suffix}_slice"][i])
                          for m in ("psnr", "ssim", "nrmse") if f"{m}{suffix}_slice" in f}
        out["attrs"] = {k: f.attrs[k] for k in ("sigma", "seed", "R", "ckpt", "task")
                        if k in f.attrs}
    return out


def slice_pairs(run):
    """Every (volume, source slice) a run has."""
    pairs = set()
    for d in _dirs(run):
        for fn in os.listdir(d):
            if fn.endswith(".h5"):
                with h5py.File(os.path.join(d, fn), "r") as f:
                    pairs |= {(fn[:-3], int(s)) for s in f["slice_index"][()]}
    return pairs


runs = list(dict.fromkeys(r for _, r, _ in METHODS))
pairs = {r: slice_pairs(r) for r in runs}
dumped = [r for r in runs if pairs[r]]
print("have images :", dumped)
print("no dump yet :", [r for r in runs if r not in dumped])
if not dumped:
    raise SystemExit("no run has a dump under RUNS_ROOT: run bigpurple/guided_recon_dump.sbatch")

if SAMPLES is None:
    common = set.intersection(*(pairs[r] for r in dumped))
    rng = random.Random(SAMPLE_SEED)
    vols = sorted({v for v, _ in common})
    rng.shuffle(vols)                       # one slice per volume, spread over patients
    SAMPLES = [(v, rng.choice(sorted(s for vv, s in common if vv == v)))
               for v in vols[:N_SAMPLES]]
print("samples     :", SAMPLES)

In [ ]:
# Runs without these slices: compute them here, through dump_eval's own code path.
todo = [r for r in runs if not all(find(r, v, s) for v, s in SAMPLES)]
for r in [r for r in todo if not os.path.exists(os.path.join(RUNS_ROOT, r, CKPT))]:
    print(f"[skip] {r}: no dump and no {CKPT} in {os.path.join(RUNS_ROOT, r)}")
    todo.remove(r)
if todo and COMPUTE_MISSING:
    import argparse
    import torch
    sys.path.insert(0, os.path.join(REPO, "scripts"))
    import dump_eval as de
    from evaluation.metrics import build_metrics
    from evaluation.tasks import EVAL_KSPACE_TYPE, default_sigma, force_organ_mask

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    metrics = build_metrics(["psnr", "ssim", "nrmse"])
    keep = {}
    for v, s in SAMPLES:
        keep.setdefault(v, set()).add(s)
    lo, hi = min(s for _, s in SAMPLES), max(s for _, s in SAMPLES) + 1
    for run in todo:
        run_dir = os.path.join(RUNS_ROOT, run)
        cfg, model, n_params, step = de.load_run(run_dir, device, CKPT)
        force_organ_mask(cfg)                                   # as dump_eval's default
        cfg.setdefault("training", {})["val_kspace_type"] = EVAL_KSPACE_TYPE
        if cfg["task"] == "i2sb_recon":                         # the run's own sampler, + no-CG
            cfg.setdefault("eval", {}).update(also_nodc=True, inversion=False)
        args = argparse.Namespace(slices=f"{lo}:{hi}", workers=2, seed=SEED, limit=None,
                                  ckpt=CKPT, keep_slices=keep)
        n = de.dump_run(run_dir, cfg, model, n_params, step, args, sorted(keep), metrics,
                        default_sigma(cfg), os.path.join(CACHE, run, "eval_dump"))
        print(f"{run}: computed {n} slices ({cfg['task']})")
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
elif todo:
    print("not computed (COMPUTE_MISSING is off):", todo)

shown = [(lab, r, ds) for lab, r, ds in METHODS if all(find(r, v, s) for v, s in SAMPLES)]
print("columns:", [lab for lab, _, _ in shown])

In [ ]:
def prep(a):
    a = np.abs(a)
    return np.flipud(a) if FLIP else a


def crop_box(mask, pad=6):
    """Row/column slice of the anatomy's bounding box (+ pad), for CROP."""
    if not CROP or mask is None or not mask.any():
        return slice(None), slice(None)
    rows, cols = np.where(mask)
    return (slice(max(rows.min() - pad, 0), rows.max() + pad + 1),
            slice(max(cols.min() - pad, 0), cols.max() + pad + 1))


def window(ref, mask):
    """Display ceiling shared by every column of a row: the viewer's (99.9th pct x 0.85)."""
    v = ref[mask] if mask is not None and mask.any() else ref
    return float(np.percentile(v, 99.9)) * 0.85


# every slice once: the shared images from the first shown run, each column's own image
base = shown[0][1]
rows = []
for v, s in SAMPLES:
    b = read(base, v, s)
    mask = prep(b["organ_mask"]).astype(bool) if "organ_mask" in b else None
    row = dict(name=f"{v} / {s}", ref=prep(b["reference"]), mask=mask,
               zf=prep(b["zero_filled"]) if "zero_filled" in b else None,
               guide=prep(b["guide"]) if "guide" in b else None, cols=[])
    for lab, r, ds in shown:
        x = read(r, v, s, ds)
        row["cols"].append((lab, prep(x["img"]), x["metrics"], x["attrs"]))
    rows.append(row)

# columns must be the same problem: warn if a run was dumped at another sigma / seed / R
for row in rows:
    a0 = row["cols"][0][3]
    for lab, _, _, a in row["cols"][1:]:
        bad = [k for k in ("sigma", "seed", "R") if k in a0 and k in a and a0[k] != a[k]]
        if bad:
            print(f"[warn] {lab} differs from {row['cols'][0][0]} in {bad}")
    break

## Reconstructions
Each row one slice, each column one method; one display window per row (the reference's). Under each method: PSNR / SSIM against the reference (inside the anatomy), as stored by the dump.

In [ ]:
extra = [("Zero-filled", "zf"), ("Guide (PD)", "guide")]
extra = [(lab, k) for lab, k in extra if rows[0][k] is not None]
ncol = len(extra) + len(shown) + 1
r0, c0 = crop_box(rows[0]["mask"])
h, w = rows[0]["ref"][r0, c0].shape
fig, axes = plt.subplots(len(rows), ncol, figsize=(1.9 * ncol, 1.9 * h / w * len(rows) + 0.6),
                         squeeze=False)
for i, row in enumerate(rows):
    rs, cs = crop_box(row["mask"])
    vmax = window(row["ref"], row["mask"])
    show = lambda ax, a: ax.imshow(np.clip(a[rs, cs] / vmax, 0, 1) ** 0.85, cmap="gray",  # noqa: E731
                                   vmin=0, vmax=1, interpolation="nearest")
    panels = ([(lab, row[k], None) for lab, k in extra]
              + [(lab, img, m) for lab, img, m, _ in row["cols"]]
              + [("Ground truth", row["ref"], None)])
    for j, (lab, img, m) in enumerate(panels):
        ax = axes[i, j]
        show(ax, img)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(lab, fontsize=9)
        if j == 0:
            ax.set_ylabel(row["name"], fontsize=7)
        if m:
            ax.set_xlabel(f"{m.get('psnr', np.nan):.2f} dB / {m.get('ssim', np.nan):.3f}",
                          fontsize=7)
plt.tight_layout()
plt.show()

## Residuals
|reference - method|, x `RESID_GAIN`, on the same per-row window (so a residual that fills the colour range is 1/RESID_GAIN of the image's brightness). Under each: NRMSE.

In [ ]:
rcols = ([("Zero-filled", "zf")] if rows[0]["zf"] is not None else [])
ncol = len(rcols) + len(shown)
fig, axes = plt.subplots(len(rows), ncol, figsize=(1.9 * ncol, 1.9 * h / w * len(rows) + 0.6),
                         squeeze=False)
for i, row in enumerate(rows):
    rs, cs = crop_box(row["mask"])
    vmax = window(row["ref"], row["mask"])
    panels = ([(lab, row[k], None) for lab, k in rcols]
              + [(lab, img, m) for lab, img, m, _ in row["cols"]])
    for j, (lab, img, m) in enumerate(panels):
        ax = axes[i, j]
        res = np.abs(row["ref"] - img)[rs, cs] * RESID_GAIN / vmax
        ax.imshow(np.clip(res, 0, 1), cmap="inferno", vmin=0, vmax=1, interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f"{lab}", fontsize=9)
        if j == 0:
            ax.set_ylabel(row["name"], fontsize=7)
        if m:
            ax.set_xlabel(f"NRMSE {m.get('nrmse', np.nan):.4f}", fontsize=7)
fig.suptitle(f"|reference - recon| x {RESID_GAIN:g}", fontsize=10)
plt.tight_layout()
plt.show()

## Numbers
Mean stored metrics over the shown slices, and over every slice each run's full dump holds (dumped runs only; computed-here runs have just the shown slices).

In [ ]:
def mean_metrics(run, ds, pairs_):
    vals = {"psnr": [], "ssim": [], "nrmse": []}
    for v, s in pairs_:
        hit = find(run, v, s)
        if hit is None:
            continue
        for k, x in read(run, v, s, ds)["metrics"].items():
            vals[k].append(x)
    return {k: (np.mean(x) if x else np.nan) for k, x in vals.items()}, len(vals["psnr"])


full = set.intersection(*(slice_pairs(r) for r in dumped)) if dumped else set()
print(f"{'method':<22}{'shown: psnr':>12}{'ssim':>8}{'nrmse':>9}   "
      f"{'all dumped: psnr':>17}{'ssim':>8}{'n':>5}")
for lab, r, ds in shown:
    a, _ = mean_metrics(r, ds, SAMPLES)
    b, n = mean_metrics(r, ds, sorted(full)) if r in dumped else ({}, 0)
    print(f"{lab:<22}{a['psnr']:12.2f}{a['ssim']:8.4f}{a['nrmse']:9.4f}   "
          + (f"{b['psnr']:17.2f}{b['ssim']:8.4f}{n:5d}" if n else f"{'-':>17}"))

## I2SB-Recon: tuning the data-consistency weight

Without CG the I2SB sampler never sees the PDFS measurement -- the bridge UNet takes only x_t (and
the guide, for `_x1`), so "no CG" is PD -> PDFS synthesis. CG is where the measurement enters:
every x0 prediction x_hat is replaced by

    argmin_x ||y - E x||^2 + lam ||x - x_hat||^2        (DC_ITERS CG steps, started at x_hat)

before the bridge step. Small `lam` fits the measured k-space exactly (the runs use 7e-5, from
I2SB-Inversion); large `lam` keeps the net's guess. E's normal operator E^H E is a k-space mask
under the coil maps, so with an orthonormal FFT and unit-norm maps its eigenvalues are at most 1:
`lam` ~ 1 is an even split, and `lam` >> 1 is "no CG".

Each (run, lam) is computed on the `SAMPLES` slices through the dump's own code path and cached in
`_compare_cache/<run>/dc<iters>_lam<lam>[_nfe<n>]/`; re-running computes only settings it has not
seen. Every setting of a run draws the same bridge noise (the dump's seed), so the columns differ
in `lam` alone. Metrics: the dump's, PSNR / SSIM with the peak fixed at 1, inside the organ mask.

In [ ]:
# ---- the knobs
DC_RUNS = [("I2SB",         "sbunet_small_R8_g1_tau0.19"),
           ("I2SB + guide", "sbunet_small_x1_R8_g1_tau0.19")]
DC_LAMS = [7e-5, 1e-3, 1e-2, 1e-1, 1.0]     # the CG weight lam; 7e-5 = what the runs use
DC_ITERS = 7                                 # CG steps per prediction (the runs use 7)
DC_NFE = None                                # sampler steps; None = each run's own val_nfe
DC_REFERENCE = ("LGGS guide-W", "lggs_guide_w_R8_g1")   # an end-to-end column to compare, or None


def dc_dir(run, iters, lam):
    """Cache directory of one setting; iters 0 = the plain sampler (no CG)."""
    tag = "nodc" if iters == 0 else f"dc{iters}_lam{lam:.0e}"
    tag += f"_nfe{DC_NFE}" if DC_NFE else ""
    return os.path.join(CACHE, run, tag, "eval_dump")


def dc_find(d, vol, sl):
    p = os.path.join(d, vol + ".h5")
    if not os.path.exists(p):
        return None
    with h5py.File(p, "r") as f:
        idx = [int(s) for s in f["slice_index"][()]]
    return (p, idx.index(sl)) if sl in idx else None


def dc_read(d, vol, sl):
    p, i = dc_find(d, vol, sl)
    with h5py.File(p, "r") as f:
        out = {k: f[k][i] for k in ("reference", "recon", "organ_mask") if k in f}
        out["metrics"] = {m: float(f[f"{m}_slice"][i])
                          for m in ("psnr", "ssim", "nrmse") if f"{m}_slice" in f}
    return out


def dc_settings():
    return [(0, 0.0)] + [(DC_ITERS, float(lam)) for lam in DC_LAMS]


# ---- compute what is missing (GPU)
import argparse
import torch
sys.path.insert(0, os.path.join(REPO, "scripts"))
import dump_eval as de
from evaluation.metrics import build_metrics
from evaluation.tasks import EVAL_KSPACE_TYPE, default_sigma, force_organ_mask

dc_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dc_metrics = build_metrics(["psnr", "ssim", "nrmse"])
dc_keep = {}
for v, s in SAMPLES:
    dc_keep.setdefault(v, set()).add(s)
dc_lo, dc_hi = min(s for _, s in SAMPLES), max(s for _, s in SAMPLES) + 1
DC_DONE = []
for lab, run in DC_RUNS:
    todo = [(it, lam) for it, lam in dc_settings()
            if not all(dc_find(dc_dir(run, it, lam), v, s) for v, s in SAMPLES)]
    run_dir = os.path.join(RUNS_ROOT, run)
    if todo and not os.path.exists(os.path.join(run_dir, CKPT)):
        print(f"[skip] {run}: no {CKPT} in {run_dir}")
        continue
    if todo:
        cfg, model, n_params, step = de.load_run(run_dir, dc_dev, CKPT)
        force_organ_mask(cfg)                                   # as dump_eval's default
        cfg.setdefault("training", {})["val_kspace_type"] = EVAL_KSPACE_TYPE
        args = argparse.Namespace(slices=f"{dc_lo}:{dc_hi}", workers=2, seed=SEED, limit=None,
                                  ckpt=CKPT, keep_slices=dc_keep)
        for it, lam in todo:
            cfg["eval"] = dict(dc_iters=it, dc_lam=lam, also_nodc=False, inversion=False,
                               **({"nfe": int(DC_NFE)} if DC_NFE else {}))
            n = de.dump_run(run_dir, cfg, model, n_params, step, args, sorted(dc_keep),
                            dc_metrics, default_sigma(cfg), dc_dir(run, it, lam))
            print(f"{run}: " + ("no CG" if it == 0 else f"lam {lam:g}, {it} CG iters")
                  + f" -> {n} slices")
        del model
        if dc_dev.type == "cuda":
            torch.cuda.empty_cache()
    DC_DONE.append((lab, run))
print("ready:", [lab for lab, _ in DC_DONE])

In [ ]:
# ---- images: per run, rows = samples, columns = no CG, each lam (, the reference), ground truth
DC_RESIDUALS = True       # also the |reference - recon| x RESID_GAIN figure


def dc_columns(run):
    cols = [("no CG", dc_dir(run, 0, 0.0))]
    cols += [(f"lam {lam:g}", dc_dir(run, DC_ITERS, float(lam))) for lam in DC_LAMS]
    return cols


ref_ok = DC_REFERENCE is not None and all(find(DC_REFERENCE[1], v, s) for v, s in SAMPLES)
for lab, run in DC_DONE:
    cols = dc_columns(run)
    for resid in ((False, True) if DC_RESIDUALS else (False,)):
        n_c = len(cols) + int(ref_ok) + (0 if resid else 1)
        fig, axs = plt.subplots(len(SAMPLES), n_c, figsize=(2.1 * n_c, 3.2 * len(SAMPLES)),
                                squeeze=False)
        for r, (v, s) in enumerate(SAMPLES):
            base = dc_read(cols[0][1], v, s)
            ref = prep(base["reference"])
            mask = prep(base["organ_mask"]).astype(bool) if "organ_mask" in base else None
            rs, cs = crop_box(mask)
            vmax = window(ref, mask)
            panels = [(name, dc_read(d, v, s)) for name, d in cols]
            if ref_ok:
                x = read(DC_REFERENCE[1], v, s)
                panels.append((DC_REFERENCE[0], {"recon": x["img"], "metrics": x["metrics"]}))
            for c, (name, x) in enumerate(panels):
                im = prep(x["recon"])
                im = np.abs(im - ref) * RESID_GAIN if resid else im
                a = axs[r, c]
                a.imshow(im[rs, cs], cmap="inferno" if resid else "gray", vmin=0, vmax=vmax)
                m = x["metrics"]
                a.set_xlabel(f"{m.get('nrmse', np.nan):.3f}" if resid else
                             f"{m.get('psnr', np.nan):.2f} dB / {m.get('ssim', np.nan):.3f}",
                             fontsize=8)
                if r == 0:
                    a.set_title(name, fontsize=9)
            if not resid:
                axs[r, -1].imshow(ref[rs, cs], cmap="gray", vmin=0, vmax=vmax)
                if r == 0:
                    axs[r, -1].set_title("Ground truth", fontsize=9)
            axs[r, 0].set_ylabel(f"{v} / {s}", fontsize=8)
        for a in axs.flat:
            a.set_xticks([]), a.set_yticks([])
        fig.suptitle(f"{lab} ({run}): " + (f"|ground truth - recon| x{RESID_GAIN:g}, NRMSE"
                                          if resid else "CG weight lam"), fontsize=11)
        plt.tight_layout()
        plt.show()

In [ ]:
# ---- metrics vs lam (mean over SAMPLES)
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
print(f"{'run':<16}{'setting':<14}{'psnr':>8}{'ssim':>8}{'nrmse':>8}")
for k, (lab, run) in enumerate(DC_DONE):
    rows = []
    for it, lam in dc_settings():
        ms = [dc_read(dc_dir(run, it, lam), v, s)["metrics"] for v, s in SAMPLES]
        mean = {key: float(np.mean([m[key] for m in ms])) for key in ("psnr", "ssim", "nrmse")}
        rows.append((it, lam, mean))
        print(f"{lab:<16}{('no CG' if it == 0 else f'lam {lam:g}'):<14}"
              f"{mean['psnr']:8.2f}{mean['ssim']:8.4f}{mean['nrmse']:8.4f}")
    color = f"C{k}"
    for a, key in zip(axs, ("psnr", "ssim", "nrmse")):
        a.semilogx([lam for it, lam, _ in rows if it], [m[key] for it, _, m in rows if it], "o-",
                   color=color, label=lab)
        a.axhline(rows[0][2][key], color=color, ls=":", label=f"{lab}, no CG")
if ref_ok:
    ms = [read(DC_REFERENCE[1], v, s)["metrics"] for v, s in SAMPLES]
    for a, key in zip(axs, ("psnr", "ssim", "nrmse")):
        a.axhline(np.mean([m[key] for m in ms]), color="k", ls="--", label=DC_REFERENCE[0])
for a, key in zip(axs, ("PSNR (dB, peak 1)", "SSIM", "NRMSE")):
    a.set(xlabel="CG weight lam", ylabel=key)
    a.grid(alpha=0.3, which="both")
axs[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## One I2SB run, step by step

The sampler's every step on one slice, next to the net's guess on a training-style input at the
same step. Set `DIAG_DC = (7, 1e-2)` (iterations, lam) to look at another CG weight.

In [ ]:
# ---- One I2SB-Recon run on one slice, step by step ------------------------------------------
# The dump's own code path (evaluation/tasks.py::i2sb_recon), with the sampler's every step
# recorded. Next to each step: the net's guess on a TRAINING-style input at the same step
# ("teacher-forced": x_t drawn from the forward bridge between the true x0 and x1, as in training).
#   teacher-forced good, sampler bad -> the sampler walks off the states the net was trained on
#   both bad                         -> the net / its inputs (scale, guide, sigma) are the problem
# "state / expected" = rms(x_t - bridge mean) / (kappa * std_sb): ~1 if the sampler's state looks
# like a training input at that step (the first step is x1 itself, so ~0 there by construction).
# Run it after the cells above (it uses SAMPLES, RUNS_ROOT, CKPT, SEED, prep, crop_box, window).
DIAG_RUN = "sbunet_small_R8_g1_tau0.19"
DIAG_SAMPLE = None      # (volume, slice); None = the first of SAMPLES
DIAG_NFE = None         # None = the run's own val_nfe
DIAG_DC = None          # None = the run's own (dc_iters, dc_lam), e.g. (7, 1e-2); (0, 0) = no CG
DIAG_ROWS = 6           # sampler steps shown as image rows

import torch
sys.path.insert(0, os.path.join(REPO, "scripts"))
import dump_eval as de
import sb.i2sb_recon as sbr
import training.i2sb_recon as tir
from datasets.registry import build_loader
from evaluation.metrics import nrmse as m_nrmse, psnr as m_psnr
from evaluation.tasks import EVAL_KSPACE_TYPE, build_adapter, default_sigma, force_organ_mask
from sb.base import bridge_coeffs, forward_sample, forward_std

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vol, sl = DIAG_SAMPLE or SAMPLES[0]
cfg, net, _, ckpt_step = de.load_run(os.path.join(RUNS_ROOT, DIAG_RUN), dev, CKPT)
net.eval()
assert cfg["task"] == "i2sb_recon", cfg["task"]
force_organ_mask(cfg)
cfg.setdefault("training", {})["val_kspace_type"] = EVAL_KSPACE_TYPE
ev = cfg.setdefault("eval", {})
ev.update(also_nodc=False, inversion=False)
if DIAG_NFE:
    ev["nfe"] = int(DIAG_NFE)
if DIAG_DC is not None:
    ev["dc_iters"], ev["dc_lam"] = int(DIAG_DC[0]), float(DIAG_DC[1])

val = dict(cfg["data"]["val"])
val.update(enumerate_slices=True, start_slice=sl, end_slice=sl + 1, volumes=[vol],
           batch_size=1, num_workers=0)
batch = next(iter(build_loader(val, shuffle=False, drop_last=False)))

# capture what the adapter measures and hands the sampler; record every sampler step
cap, trace, extras = {}, [], {}
real_measure, real_sample = tir._measure, sbr.i2sb_recon_sample


def _measure_rec(*a, **k):
    out = real_measure(*a, **k)
    cap.update(zip(("x0", "x1", "y", "E", "sigma_n", "_om", "_gt", "guide"), out))
    return out


def _sample_rec(*a, **k):
    cap.update(sched=a[3], kappa=k.get("noise_scale"))
    return real_sample(*a, **dict(k, trace=trace))


tir._measure, sbr.i2sb_recon_sample = _measure_rec, _sample_rec
try:
    with torch.no_grad():
        gt, rec, omask = build_adapter("i2sb_recon")(
            net, batch, cfg, dev, default_sigma(cfg),
            torch.Generator(device=dev).manual_seed(SEED), extras)
finally:
    tir._measure, sbr.i2sb_recon_sample = real_measure, real_sample

sched, kappa, x0, x1 = cap["sched"], cap["kappa"], cap["x0"], cap["x1"]
kap = 1.0 if kappa is None else float(torch.as_tensor(kappa).flatten()[0])
mu0, mu1, std_sb = bridge_coeffs(sched)
M = cap["_om"]                                     # the measurement's organ mask (metric region)
M = None if M is None else M.to(dev).float()


def rms(a):
    a = a.abs() ** 2
    return float((a * M).sum() / M.sum() if M is not None else a.mean()) ** 0.5


def rel(a, b):
    """relative error inside the organ mask, complex"""
    return rms(a - b) / rms(b)


steps = []
torch.manual_seed(0)
with torch.no_grad():
    for t in trace:
        n = t["step"]
        st = torch.full((1,), n, device=dev, dtype=torch.long)
        xt = t["x_t"].to(dev)
        exp_std = kap * float(std_sb[n])
        z = forward_sample(sched, st, x0, x1, noise_scale=kappa)
        tf = sbr.predict_x0_recon(net, z, forward_std(sched, st, xdim=x0.shape[1:]), x1=x1,
                                  y=cap["y"], E=cap["E"], sigma_n=cap["sigma_n"],
                                  guide=cap["guide"])
        steps.append(dict(
            step=n, sigma=float(sched.std_fwd[n]),
            state=rms(xt - (mu0[n] * x0 + mu1[n] * x1)) / exp_std if exp_std > 0 else np.nan,
            pre=rel(t["x0_pre"].to(dev), x0), dc=rel(t["x0_dc"].to(dev), x0), tf=rel(tf, x0),
            tf_img=tf.cpu()))

# ---- numbers
print(f"{DIAG_RUN} @ step {ckpt_step}, {vol} / {sl}: nfe {len(trace)}, "
      f"dc {ev.get('dc_iters', cfg['i2sb'].get('dc_iters', 0))} iters, "
      f"x1_source {cfg['i2sb'].get('x1_source', 'adjoint')}, "
      f"cond_x1 {getattr(net, 'cond_x1', '?')}, kappa {kap:.3g}")
print(f"rms in mask: x0 (target) {rms(x0):.4g}   gt {rms(gt):.4g}   x1 (start) {rms(x1):.4g}   "
      f"zero-filled {rms(extras['zero_filled']):.4g}   sigma_n {float(torch.as_tensor(cap['sigma_n']).flatten()[0]):.3g}")
print(f"rel. error: x1 vs x0 {rel(x1, x0):.3f}   x0 vs gt {rel(x0, gt):.3f}   "
      f"zero-filled vs x0 {rel(extras['zero_filled'], x0):.3f}")
print(f"\n{'step':>5}{'sigma':>8}{'state/expected':>16}{'net guess':>11}{'after CG':>10}"
      f"{'teacher-forced':>16}     (rel. error vs x0)")
for s in steps:
    print(f"{s['step']:5d}{s['sigma']:8.3f}{s['state']:16.2f}{s['pre']:11.3f}{s['dc']:10.3f}"
          f"{s['tf']:16.3f}")
p = float(m_psnr(gt, rec, omask)[0])
print(f"\nfinal recon: PSNR {p:.2f} dB, NRMSE {float(m_nrmse(gt, rec, omask)[0]):.4f} (vs gt)")
if find(DIAG_RUN, vol, sl):
    print(f"the dump has PSNR {read(DIAG_RUN, vol, sl)['metrics'].get('psnr', np.nan):.2f} dB "
          f"on this slice (differs only in bridge-noise draw / sampler overrides)")

# ---- curves
k = np.arange(len(steps))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for key, lab in (("pre", "sampler: net guess"), ("dc", "sampler: after CG"),
                 ("tf", "teacher-forced net guess")):
    ax[0].plot(k, [s[key] for s in steps], "o-", label=lab)
ax[0].axhline(rel(x1, x0), color="k", ls=":", label="bridge start x1")
ax[0].set(ylabel="rel. error vs x0", title="x0 guesses along the sampler")
ax[1].plot(k, [s["state"] for s in steps], "o-")
ax[1].axhline(1, color="k", ls=":")
ax[1].set(ylabel="rms(x_t - bridge mean) / expected std", title="is x_t a training-like state?")
for a in ax:
    a.set_xticks(k, [f"{s['sigma']:.2f}" for s in steps], rotation=60, fontsize=7)
    a.set_xlabel("sigma (sampler order, left = start)")
    a.grid(alpha=0.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

# ---- images: rows = sampler steps
np_ = lambda t: prep(t.detach().cpu().numpy().squeeze())          # noqa: E731
mask_np = prep(M.cpu().numpy().squeeze()).astype(bool) if M is not None else None
rs, cs = crop_box(mask_np)
ref_np = np_(x0)
vmax = window(ref_np, mask_np)
pick = sorted(set(np.linspace(0, len(steps) - 1, min(DIAG_ROWS, len(steps))).round().astype(int)))
cols = ["state x_t", "net guess", "after CG", "teacher-forced guess",
        f"|after CG - x0| x{RESID_GAIN:g}"]
fig, axs = plt.subplots(len(pick), len(cols), figsize=(2.3 * len(cols), 2.6 * len(pick)),
                        squeeze=False)
for r, i in enumerate(pick):
    t, s = trace[i], steps[i]
    imgs = [np_(t["x_t"]), np_(t["x0_pre"]), np_(t["x0_dc"]), np_(s["tf_img"]),
            np_(t["x0_dc"].to(dev) - x0) * RESID_GAIN]
    errs = [s["state"], s["pre"], s["dc"], s["tf"], None]
    for c, (im, e) in enumerate(zip(imgs, errs)):
        a = axs[r, c]
        a.imshow(im[rs, cs], cmap="inferno" if c == 4 else "gray", vmin=0, vmax=vmax)
        a.set_xticks([]), a.set_yticks([])
        if e is not None:
            a.text(0.02, 0.02, (f"x{e:.2f} std" if c == 0 else f"{e:.3f}"), color="yellow",
                   fontsize=8, transform=a.transAxes)
        if r == 0:
            a.set_title(cols[c], fontsize=9)
    axs[r, 0].set_ylabel(f"step {s['step']}\nsigma {s['sigma']:.2f}", fontsize=9)
plt.tight_layout()
plt.show()

# ---- start and end
ends = [("zero-filled", np_(extras["zero_filled"])), ("bridge start x1", np_(x1)),
        ("final recon", np_(rec)), ("target x0", ref_np),
        (f"|recon - gt| x{RESID_GAIN:g}", np_(rec - gt) * RESID_GAIN)]
fig, axs = plt.subplots(1, len(ends), figsize=(2.3 * len(ends), 2.8))
for a, (lab, im) in zip(axs, ends):
    a.imshow(im[rs, cs], cmap="inferno" if lab.startswith("|") else "gray", vmin=0, vmax=vmax)
    a.set_title(lab, fontsize=9)
    a.set_xticks([]), a.set_yticks([])
plt.tight_layout()
plt.show()

del net
if dev.type == "cuda":
    torch.cuda.empty_cache()